# where-clip-negative — ex2: two-sided clamp via stacked `torch.where` calls (plus lo<=hi validation)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `where-clip-negative`. Running the final beacon cell reports progress against the `PyTorch: where to clip negative` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: where to clip negative` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`where-clip-negative`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "where-clip-negative"
DD_SUBTOPIC = "PyTorch: where to clip negative"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## three-way clamp via two `torch.where` calls

Ex1 used `torch.where` to clip negatives — a one-sided clamp. The deepening move is a TWO-SIDED clamp: clip to `[lo, hi]` using two stacked `where` calls.

```python
# Step 1: lift everything below `lo` up to `lo`.
y = t.where(x < lo, t.full_like(x, lo), x)
# Step 2: pull everything above `hi` down to `hi`.
y = t.where(y > hi, t.full_like(y, hi), y)
```

**Equivalent to `torch.clamp(x, lo, hi)`.** The exercise rebuilds it from `where` to make the control flow visible — two scalar predicates, two broadcast selects.

**Order doesn't matter when `lo <= hi`.** Either sweep can run first; the second sweep then re-clips its own output. If `lo > hi` (a typo trap) the two orders disagree — the exercise asks you to detect that case up front and raise `ValueError`.

**`torch.full_like(x, lo)` over `lo * torch.ones_like(x)`.** One op, matches dtype + device + shape automatically.

### Exercise 2 — two-sided clamp via stacked `torch.where` calls (plus lo<=hi validation)

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply two `torch.where` calls to clip a tensor to `[lo, hi]`, matching `torch.clamp`'s output, and raise `ValueError` if `lo > hi`.
> Keywords: where, clamp, two-sided
> ```

**KCs targeted:** `stacked-where-for-two-sided-clip`, `precondition-low-less-or-equal-high`

Implement `ex2_clamp_via_where(x, lo, hi)`. A two-sided clamp built from `torch.where`.

Inputs:
- `x`: float tensor of any shape.
- `lo`: float, lower bound (inclusive).
- `hi`: float, upper bound (inclusive).

Algorithm:
1. If `lo > hi`: raise `ValueError` whose message mentions both `lo` and `hi` (case-insensitive).
2. `y = torch.where(x < lo, torch.full_like(x, lo), x)` — lift the under-floor values.
3. `y = torch.where(y > hi, torch.full_like(y, hi), y)` — pull the over-ceiling values.
4. Return `y`. Same shape, same dtype as `x`.

Constraints:
- DO NOT call `torch.clamp` directly. Build it from `where`.
- DO NOT use `torch.minimum`/`torch.maximum`.
- DO NOT mutate `x`.

In [ ]:
def ex2_clamp_via_where(x: Tensor, lo: float, hi: float) -> Tensor:
    """Two-sided clamp to [lo, hi] built from two torch.where calls."""
    raise NotImplementedError()


def _test_ex2():
    # === Matches torch.clamp on a random tensor ===
    t.manual_seed(0)
    x = t.randn(64) * 5.0
    out = ex2_clamp_via_where(x, lo=-2.0, hi=3.0)
    ref = t.clamp(x, -2.0, 3.0)
    assert t.allclose(out, ref), f'must match torch.clamp; first diff at {(out - ref).abs().argmax().item()}'
    assert out.shape == x.shape and out.dtype == x.dtype

    # === Hand-traced 1-D ===
    x = t.tensor([-5.0, -1.0, 0.0, 1.0, 5.0, 10.0])
    out = ex2_clamp_via_where(x, lo=-2.0, hi=3.0)
    expected = t.tensor([-2.0, -1.0, 0.0, 1.0, 3.0, 3.0])
    assert t.equal(out, expected), f'expected={expected}, got {out}'

    # === Boundary values pass through unchanged ===
    x = t.tensor([-2.0, 3.0])
    out = ex2_clamp_via_where(x, lo=-2.0, hi=3.0)
    assert t.equal(out, x), f'boundary values must pass through, got {out}'

    # === Input not mutated ===
    x_orig = t.tensor([-10.0, 0.0, 10.0])
    x_clone = x_orig.clone()
    _ = ex2_clamp_via_where(x_clone, -1.0, 1.0)
    assert t.equal(x_clone, x_orig), 'must not mutate input'

    # === lo == hi → all values pinned to that single value ===
    x = t.tensor([-5.0, 0.0, 5.0])
    out = ex2_clamp_via_where(x, lo=2.5, hi=2.5)
    assert t.allclose(out, t.full_like(x, 2.5)), f'lo==hi must pin everything, got {out}'

    # === lo > hi → ValueError mentioning lo and hi ===
    try:
        ex2_clamp_via_where(t.tensor([0.0]), lo=5.0, hi=1.0)
    except ValueError as e:
        msg = str(e).lower()
        assert 'lo' in msg and 'hi' in msg, f'error must mention lo and hi, got {e!r}'
    else:
        raise AssertionError('expected ValueError for lo > hi')

    # === Multi-D shape preserved ===
    x = t.randn(3, 4, 5)
    out = ex2_clamp_via_where(x, -0.5, 0.5)
    assert out.shape == (3, 4, 5), f'shape wrong: {tuple(out.shape)}'
    assert (out >= -0.5).all() and (out <= 0.5).all(), 'all values must be in [lo, hi]'

    # === Negative-only range still works ===
    x = t.tensor([-10.0, -3.0, 0.0, 3.0, 10.0])
    out = ex2_clamp_via_where(x, lo=-5.0, hi=-1.0)
    expected = t.tensor([-5.0, -3.0, -1.0, -1.0, -1.0])
    assert t.equal(out, expected), f'expected={expected}, got {out}'

    # === Dtype preservation (int) ===
    # Note: full_like passes lo/hi through torch's float-to-int cast.
    x = t.tensor([-5, 0, 5, 10], dtype=t.int32)
    out = ex2_clamp_via_where(x, lo=-2, hi=3)
    assert out.dtype == x.dtype, f'int dtype must be preserved, got {out.dtype}'
    assert t.equal(out, t.tensor([-2, 0, 3, 3], dtype=t.int32))
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_clamp_via_where(x, lo, hi):
    if lo > hi:
        raise ValueError(f'lo must be <= hi, got lo={lo} hi={hi}')
    y = t.where(x < lo, t.full_like(x, lo), x)
    y = t.where(y > hi, t.full_like(y, hi), y)
    return y
```

**Two `where` calls > one nested `where`.** A nested form like `where(x<lo, lo, where(x>hi, hi, x))` works but is harder to read. The stacked form makes each clip step a separate line — easier to debug when one bound is wrong.

**`torch.full_like(x, lo)` over a scalar broadcast.** Scalars broadcast fine, but `full_like` makes the dtype + device matching explicit. On an `int32` input with `lo=1.5`, `full_like` rounds (or PyTorch will error on type mismatch) instead of silently promoting to float64.

**`lo > hi` raises; `lo == hi` is fine.** A single-point clamp is a legitimate (if degenerate) use case — projecting everything onto a constant. Only the strictly-inverted bound is the error condition.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()